# Clause Segmentation for Legal Documents

## Objective
Legal contracts are long documents.  
Before classification or risk analysis, the document must be split into **individual clauses**.

This notebook:
- Loads preprocessed legal text
- Applies clause segmentation
- Prepares clause-level data for modeling


In [1]:
import pandas as pd
import re
import nltk
from nltk.tokenize import sent_tokenize

nltk.download("punkt")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ADITYA\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [2]:
# Load preprocessed CUAD data
df = pd.read_csv("../data/processed/cuad_preprocessed.csv")

df.head()


,file_name,clause,pages,class_id,label,start_at,end_at,clean_text,sentences,word_count
0,EuromediaHoldingsCorp_20070215_10SB12G_EX-10.B...,In the event that Licensor grants to another V...,2,8,Most Favored Nation,2558,2929,in the event that licensor grants to another v...,['in the event that licensor grants to another...,60
1,EuromediaHoldingsCorp_20070215_10SB12G_EX-10.B...,"If Licensor enters, or has entered, into an ag...",8,8,Most Favored Nation,18515,19562,if licensor enters or has entered into an agre...,['if licensor enters or has entered into an ag...,169
2,EuromediaHoldingsCorp_20070215_10SB12G_EX-10.B...,"Licensor shall provide to Rogers, no later tha...",8,8,Most Favored Nation,19563,20059,licensor shall provide to rogers no later than...,['licensor shall provide to rogers no later th...,79
3,IntegrityMediaInc_20010329_10-K405_EX-10.17_23...,"If for any reason, Integrity and TL are subjec...",3,8,Most Favored Nation,8173,8345,if for any reason integrity and tl are subject...,['if for any reason integrity and tl are subje...,30
4,TomOnlineInc_20060501_20-F_EX-4.46_749700_EX-4...,"The Company will, and Online BVI will cause th...",10,8,Most Favored Nation,30765,31577,the company will and online bvi will cause the...,['the company will and online bvi will cause t...,123


## Why Clause Segmentation?

A single legal document may contain:
- Multiple obligations
- Multiple risks
- Multiple clause types

Clause-level analysis allows:
- Better classification
- Accurate risk scoring
- Clear explanations to users


In [3]:
def segment_clauses(text):
    """
    Splits legal text into clauses using rule-based patterns.
    """
    # Split on common legal separators
    clauses = re.split(
        r"\.\s+|;\s+|\n+|\:\s+", text
    )
    
    # Clean clauses
    clauses = [c.strip() for c in clauses if len(c.strip()) > 20]
    
    return clauses


In [4]:
df["clauses"] = df["clean_text"].apply(segment_clauses)

df[["clean_text", "clauses"]].head(2)


,clean_text,clauses
0,in the event that licensor grants to another v...,[in the event that licensor grants to another ...
1,if licensor enters or has entered into an agre...,[if licensor enters or has entered into an agr...


In [5]:
# Explode clauses into individual rows
clause_df = df.explode("clauses").reset_index(drop=True)

clause_df = clause_df.rename(columns={"clauses": "clause_text"})

clause_df[["clause_text", "label"]].head()


,clause_text,label
0,in the event that licensor grants to another v...,Most Favored Nation
1,if licensor enters or has entered into an agre...,Most Favored Nation
2,licensor shall provide to rogers no later than...,Most Favored Nation
3,if for any reason integrity and tl are subject...,Most Favored Nation
4,the company will and online bvi will cause the...,Most Favored Nation


In [10]:
# Remove any NaN values in clause_text
clause_df = clause_df.dropna(subset=["clause_text"])

clause_df["word_count"] = clause_df["clause_text"].apply(
    lambda x: len(str(x).split())
)

# Filter very small clauses
clause_df = clause_df[clause_df["word_count"] > 5]

print("Total clauses after segmentation:", len(clause_df))

Total clauses after segmentation: 9698


In [11]:
clause_df.sample(5)[["clause_text", "label"]]


,clause_text,label
7596,this agreement shall be governed and construed...,Governing Law
3482,except as expressly provided in this agreement...,Non-Transferable License
8492,failure of distributor to purchase the minimum...,Minimum Commitment
7897,such personal appearances shall be limited to ...,Volume Restriction
3069,this agreement grants envision a nonexclusive ...,License Grant


In [12]:
import os

os.makedirs("../data/processed", exist_ok=True)

clause_df.to_csv(
    "../data/processed/cuad_segmented_clauses.csv",
    index=False
)

print("Segmented clauses saved to data/processed/cuad_segmented_clauses.csv")


Segmented clauses saved to data/processed/cuad_segmented_clauses.csv


## Summary

In this notebook, we:
- Loaded preprocessed legal clauses
- Applied rule-based clause segmentation
- Converted documents into clause-level rows
- Removed noisy segments
- Saved clean clause-level data

Next notebook:
➡ 03_clause_classification.ipynb
